# How to use

- เปิด Playground mode ด้วยการกด `File` -> `Open in playground mode`
- พิมพ์ Prefix ของชื่อ Sheet ที่ต้องการจะตรวจในช่องข้างล่างนี้ เช่น
  - ต้องการตรวจชีต `JOHNกระทรวงการคลัง`
  - ให้พิมพ์ `JOHN`
- กด Run all และรอโค้ดทำงาน
- error_message จะปรากฎในชีต ->ให้แก้ไข error
กลับมากด Run all เพื่อเช็กว่า error ถูกแก้ไขหรือไม่ หากแก้ไขแล้ว  error_message จะหายไป
---
Note
- ในการรันครั้งแรกระบบจะขอให้ login ด้วย Google Account
- Login ด้วย account ที่มี permission แก้ไข [sheet สำหรับ validate](https://docs.google.com/spreadsheets/d/1zArSJZE1l3nuOMbPvnxNGmATLy1g1e0IjBcJkxJLFxU/edit?gid=51205943#gid=51205943?target=_blank)

In [1]:
#@title เลือก Sheet Prefix
SHEET_PREFIX = 'WEVIS' # @param {"type": "string"}

assert SHEET_PREFIX != ''

In [2]:
#@title พ.ศ. ปีงบประมาณ
BUDGET_YEAR = 2570 # @param {"type": "number"}

assert BUDGET_YEAR

# Code

In [3]:
import os
import re
import requests
import json
import pandas as pd

## Authenticate to Google

In [4]:
# Connect to Google Account & Google Sheet
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

In [5]:
from googleapiclient.discovery import build

def get_sheet_in_drive() -> dict[str, str]:
  drive_service = build('drive', 'v3')

  try:
      session = requests.get('http://172.28.0.2:9000/api/sessions').json()
      notebook_name = session[0]['name']
      print(f"Current Notebook: {notebook_name}")
  except Exception:
      # Fallback in case internal session lookup fails
      notebook_name = "Budget-Tree-Validator.ipynb"

  # 3. Locate the notebook file in Google Drive to find its parent folder ID
  results = drive_service.files().list(
      q=f"name = '{notebook_name}' and trashed = false",
      fields="files(id, name, parents)",
      supportsAllDrives=True,
      includeItemsFromAllDrives=True
  ).execute()

  files = results.get('files', [])

  if not files:
      raise FileNotFoundError(f"Could not locate '{notebook_name}' in Google Drive. Make sure it has been saved to Drive.")

  # If multiple notebooks share the same name, files[0] is used
  parent_folder_id = files[0]['parents'][0]

  # 4. Search for all Google Sheets in the same parent folder
  sheet_query = (
      f"'{parent_folder_id}' in parents and "
      f"mimeType = 'application/vnd.google-apps.spreadsheet' and "
      f"trashed = false"
  )

  sheets_result = drive_service.files().list(
      q=sheet_query,
      fields="files(id, name, webViewLink)",
      supportsAllDrives=True,
      includeItemsFromAllDrives=True
  ).execute()

  sheets = sheets_result.get('files', [])

  # 5. Output the results
  print(f"\nFound {len(sheets)} Google Sheet(s) in this folder:\n")
  for sheet in sheets:
      print(f"📄 {sheet['name']}")
      print(f"   🔗 {sheet['webViewLink']}\n")

  return sheets

In [6]:
sheet_data_file = "sheet_in_drive.json"
if not os.path.exists(sheet_data_file):
  sheets_in_drive = get_sheet_in_drive()
  with open(sheet_data_file, "w") as f:
    json.dump(sheets_in_drive, f)
else:
  with open(sheet_data_file, "r") as f:
    sheets_in_drive = json.load(f)


Found 2 Google Sheet(s) in this folder:

📄 [70/3] Budget Tree For Validation
   🔗 https://docs.google.com/spreadsheets/d/1zArSJZE1l3nuOMbPvnxNGmATLy1g1e0IjBcJkxJLFxU/edit?usp=drivesdk

📄 Excel Data Table 2570 วาระ 3
   🔗 https://docs.google.com/spreadsheets/d/11OIXslQ0TRenaZY18Nfp_vp6FnPJ5gSB2qgQAoNwVvc/edit?usp=drivesdk



## Connect to Google Sheet

In [7]:
gc = gspread.authorize(creds)

In [8]:
# Get first found sheet
validate_sheet = gc.open_by_url(sheets_in_drive[0].get('webViewLink'))
validate_sheet

<Spreadsheet '[70/3] Budget Tree For Validation' id:1zArSJZE1l3nuOMbPvnxNGmATLy1g1e0IjBcJkxJLFxU>

## Clone & Install Validation Tool

In [9]:
!rm -rf wevis-openbudget-validation-tools

In [10]:
!git clone https://github.com/wevisdemo/wevis-openbudget-validation-tools.git

Cloning into 'wevis-openbudget-validation-tools'...
remote: Enumerating objects: 419, done.
remote: Counting objects: 100% (419/419), done.
remote: Compressing objects: 100% (244/244), done.
remote: Total 419 (delta 204), reused 377 (delta 162), pack-reused 0 (from 0)
Receiving objects: 100% (419/419), 82.40 KiB | 2.06 MiB/s, done.
Resolving deltas: 100% (204/204), done.


In [11]:
!pip install -e "wevis-openbudget-validation-tools/open-budget-tree-validator"

Obtaining file:///content/wevis-openbudget-validation-tools/open-budget-tree-validator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 102.6 MB/s eta 0:00:00
  Building editable for open_budget_tree_validator (pyproject.toml) ... done
  Created wheel for open_budget_tree_validator: filename=open_budget_tree_validator-1.1.0-0.editable-py3-none-any.whl size=1504 sha256=960d037ab1101c44069fe9822ab66000b40ee25510e02f3828eeed2e933ed169
  Stored in directory: /tmp/pip-ephem-wheel-cache-aihrvv49/wheels/95/ba/0b/0bcce617f788769c2598adbe3aa243ac8a1aaa049fc1979457
Successfully built open_budget_tree_validator


In [12]:
import site

for sitepackages in site.getsitepackages():
    site.addsitedir(sitepackages)

## Validate Tree

In [13]:
from open_budget_tree_validator import BudgetTree

In [14]:
def validate_budget_df(df: pd.DataFrame) -> pd.DataFrame:
  budget_tree = BudgetTree(df, BUDGET_YEAR)

  # Check if already auto correct
  auto_correct_flag = "_at_flag.txt"
  if not os.path.exists(auto_correct_flag):
    budget_tree.auto_correct_tree()
    with open(auto_correct_flag, "w") as f:
      f.write("auto correct")

  validated_df = budget_tree.get_validate_tree()
  return validated_df

In [15]:
for sheet in validate_sheet.worksheets():

  sheet_name = sheet.title

  if not re.search(r"^" + SHEET_PREFIX, sheet_name):
    continue

  if re.search(r"^(README|PROGRESS)", sheet_name):
    continue

  # Load sheet into df
  sheet_values = sheet.get_all_values()
  df = pd.DataFrame(sheet_values[1:], columns=sheet_values[0])
  # Reset error
  df['error_message'] = ''

  budget_tree = BudgetTree(df, BUDGET_YEAR)

  validated_tree_df = budget_tree.get_validate_tree()

  # Update sheet column A with `error_message`
  sheet.update(
    [[_] for _ in validated_tree_df['error_message'].to_list()],
    'A2:A'
  )